# 🧬 Cell Tracking — ILP Submission (offline, best config: 0.648 dev-subset)

Produces `submission.csv` for the [Biohub Cell Tracking](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)
code competition using our best pipeline: **finer-XY detection + global ILP linking**
(`xy_ds=2`, `thresh_rel=0.30`, `tracksdata` ILP). Scored **0.648** on our 13-dataset local dev subset,
up from the greedy baseline's 0.534 (≈0.618 LB).

**Setup (one-time):**
1. Run `build_wheels_nb.py` (internet ON) → save its `/kaggle/working/wheels` output as a **private** dataset.
2. In THIS notebook: attach that wheel dataset **and** the competition data, set **Internet: Off**, Run All, Submit.

The ILP solver runs offline via **SCIP** (`pyscipopt`, bundled — no Gurobi/internet). A per-dataset solve
timeout guards the 12h budget. *Generated from the verified `pipeline/` package (`build_submission_ilp_nb.py`).*

### Install ILP deps (offline, from the attached wheel dataset)

In [ ]:
# --- Offline install of ILP deps (tracksdata + pyscipopt) -------------------
# Attach the wheels dataset (kaggle.com/datasets/rkoren/biohub-celltrack-ilp-wheels)
# to this notebook; with Internet OFF this installs from it via --no-index (no network).
# Locally (deps already present), this no-ops. Robust to wheels being extracted OR zipped.
import subprocess, sys, glob, os, zipfile, shutil

FLAT = "/tmp/ilp_wheels"

def _gather_wheels() -> int:
    """Collect EVERY .whl found under /kaggle/input into one flat dir (Kaggle may
    mount/extract the dataset at a nested path, and wheels can be split across dirs)."""
    os.makedirs(FLAT, exist_ok=True)
    hits = glob.glob("/kaggle/input/**/*.whl", recursive=True)
    if not hits:  # wheels delivered inside a .zip → extract first
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            try:
                zipfile.ZipFile(z).extractall("/tmp/ilp_zip")
            except Exception:
                pass
        hits = glob.glob("/tmp/ilp_zip/**/*.whl", recursive=True)
    for w in hits:
        dst = os.path.join(FLAT, os.path.basename(w))
        if not os.path.exists(dst):
            shutil.copy(w, dst)
    return len(glob.glob(f"{FLAT}/*.whl"))

# Install our wheel ONLY for packages Kaggle doesn't already have (or has too old for tracksdata).
# Overriding Kaggle's numpy/numba/etc. with our differently-compiled builds ABI-breaks its stack, so
# we DELETE any wheel whose package is already installed — EXCEPT this KEEP set, which tracksdata needs
# at versions newer/older than Kaggle ships (polars>=1.36 vs 1.35; numcodecs<0.16; the git-only packages).
_KEEP = {"tracksdata", "pyscipopt", "ilpy", "geff", "geff_spec", "bidict", "donfig",
         "rustworkx", "polars", "polars_runtime_32", "numcodecs", "zarr", "imagecodecs"}

def _norm(whl: str) -> str:
    return whl.split("-")[0].lower().replace("-", "_")

def _installed(name: str) -> bool:
    from importlib.metadata import version, PackageNotFoundError
    for cand in (name, name.replace("_", "-")):
        try:
            version(cand); return True
        except PackageNotFoundError:
            continue
    return False

def _ensure_ilp_deps():
    try:
        import tracksdata, pyscipopt  # noqa: F401
        print("ILP deps already present — skipping install")
        return
    except ImportError:
        pass
    n = _gather_wheels()
    if n == 0:
        raise RuntimeError("No .whl found under /kaggle/input — attach the wheels dataset.")
    deferred = []
    for w in glob.glob(f"{FLAT}/*.whl"):
        name = _norm(os.path.basename(w))
        if name not in _KEEP and _installed(name):
            os.remove(w); deferred.append(name)
    kept = len(glob.glob(f"{FLAT}/*.whl"))
    print(f"installing {kept} wheels; deferring {len(deferred)} already on Kaggle (incl. "
          f"{', '.join(x for x in ('numpy','scipy','numba','llvmlite') if x in deferred)})")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--pre",
                    f"--find-links={FLAT}", "tracksdata", "pyscipopt"], check=True)

_ensure_ilp_deps()

### Imports

In [ ]:
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import tracksdata as td

### Configuration

In [ ]:
"""Pipeline configuration, sourced from menu.yaml.

Every tunable knob lives here as one dataclass so experiments are a single
object to log, diff, and sweep. Defaults match the starter's `v2_precision`
profile (public LB ≈ 0.581).
"""

from dataclasses import dataclass, field, asdict
from pathlib import Path


@dataclass
class PipelineConfig:
    # physical voxel scale, microns per voxel (z, y, x)
    scale: tuple[float, float, float] = (1.625, 0.40625, 0.40625)

    # --- detection ---
    xy_ds: int = 4                    # XY block-mean factor
    smooth_sigma: float = 0.95
    min_peak_dist: int = 3
    thresh_rel: float = 0.34
    min_rel_contrast: float = 0.08
    nms_radius_um: float = 2.65
    border_z: int = 1
    border_yx: int = 2
    border_keep_quantile: float = 0.70
    max_frame_count_mult: float = 1.70
    max_frame_count_add: int = 45
    max_nodes_per_frame: int = 20000

    # --- linking ---
    max_link_dist_um: float = 11.0
    link_method: str = "greedy"       # "greedy" (Hungarian) | "ilp" (global, tracksdata)
    link_n_neighbors: int = 6         # candidate neighbours per node for ILP
    link_delta_t: int = 1             # ILP gap-closing: also link t→t+Δ (Δ>1 bridges missed detections)
    ilp_appearance: float = 0.1
    ilp_disappearance: float = 0.1
    ilp_division: float = 1.0
    ilp_timeout: float = 600.0        # per-dataset ILP solve timeout (s) — guards the 12h budget

    # --- divisions ---
    detect_divisions: bool = True
    div_parent_dist_um: float = 8.75
    div_sister_dist_um: float = 6.25
    div_min_count_gain: int = 1

    # --- node pruning ---
    prune_isolated_nodes: bool = True
    keep_strong_isolated: bool = False
    strong_isolated_quantile: float = 0.97

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_menu(cls, menu: dict) -> "PipelineConfig":
        """Build from a parsed menu.yaml dict (scale/detect/link/divisions/prune)."""
        sc = menu.get("scale", {})
        d = menu.get("detect", {})
        lk = menu.get("link", {})
        dv = menu.get("divisions", {})
        pr = menu.get("prune", {})
        return cls(
            scale=(sc.get("z", 1.625), sc.get("y", 0.40625), sc.get("x", 0.40625)),
            xy_ds=d.get("xy_ds", 4),
            smooth_sigma=d.get("smooth_sigma", 0.95),
            min_peak_dist=d.get("min_peak_dist", 3),
            thresh_rel=d.get("thresh_rel", 0.34),
            min_rel_contrast=d.get("min_rel_contrast", 0.08),
            nms_radius_um=d.get("nms_radius_um", 2.65),
            border_z=d.get("border_z", 1),
            border_yx=d.get("border_yx", 2),
            border_keep_quantile=d.get("border_keep_quantile", 0.70),
            max_frame_count_mult=d.get("max_frame_count_mult", 1.70),
            max_frame_count_add=d.get("max_frame_count_add", 45),
            max_nodes_per_frame=d.get("max_nodes_per_frame", 20000),
            max_link_dist_um=lk.get("max_link_dist_um", 11.0),
            link_method=lk.get("method", "greedy"),
            link_n_neighbors=lk.get("n_neighbors", 6),
            link_delta_t=lk.get("delta_t", 1),
            ilp_appearance=lk.get("ilp_appearance", 0.1),
            ilp_disappearance=lk.get("ilp_disappearance", 0.1),
            ilp_division=lk.get("ilp_division", 1.0),
            ilp_timeout=lk.get("ilp_timeout", 600.0),
            detect_divisions=dv.get("enabled", True),
            div_parent_dist_um=dv.get("parent_dist_um", 8.75),
            div_sister_dist_um=dv.get("sister_dist_um", 6.25),
            div_min_count_gain=dv.get("min_count_gain", 1),
            prune_isolated_nodes=pr.get("isolated_nodes", True),
            keep_strong_isolated=pr.get("keep_strong_isolated", False),
            strong_isolated_quantile=pr.get("strong_isolated_quantile", 0.97),
        )

    @classmethod
    def load(cls, menu_path: str | Path = "menu.yaml") -> "PipelineConfig":
        import yaml
        with open(menu_path) as f:
            return cls.from_menu(yaml.safe_load(f))

### Detection

In [ ]:
"""Cell detection in a single 3D volume.

Full-Z, XY block-mean → smooth → robust threshold → local-maxima peaks →
intensity-weighted centroid refinement → physical NMS → border/count guards.

The design bias is *precision + accurate centroids*: the metric matches nodes to
sparse GT with a hard 7µm tolerance (localization is effectively binary), and
over-predicting past the dense node count only costs. So we keep stable centres,
not every bright speck.
"""

import numpy as np
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree


try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
    _SKIMAGE = True
except Exception:  # pragma: no cover
    peak_local_max = None
    threshold_otsu = None
    _SKIMAGE = False


def block_mean_xy(vol: np.ndarray, factor: int) -> np.ndarray:
    """Average-pool XY by ``factor`` while preserving Z resolution."""
    Z, Y, X = vol.shape
    Y2, X2 = (Y // factor) * factor, (X // factor) * factor
    x = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return x.reshape(Z, Y2 // factor, factor, X2 // factor, factor).mean(axis=(2, 4))


def robust_threshold(sm: np.ndarray, thresh_rel: float) -> tuple[float, float, float]:
    """Otsu + relative-rise threshold. Returns (threshold, background, dyn_range)."""
    bg = float(np.median(sm))
    hi = float(np.percentile(sm, 99.9))
    dyn = max(hi - bg, 1e-6)
    rel_thr = bg + thresh_rel * dyn
    try:
        otsu = float(threshold_otsu(sm)) if _SKIMAGE else float(np.percentile(sm, 96.0))
    except Exception:
        otsu = float(np.percentile(sm, 96.0))
    return max(otsu, rel_thr), bg, dyn


def _fallback_peaks(sm: np.ndarray, threshold_abs: float, min_distance: int) -> np.ndarray:
    size = 2 * int(min_distance) + 1
    mx = maximum_filter(sm, size=(size, size, size), mode="nearest")
    mask = (sm >= mx) & (sm > threshold_abs)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return coords.astype(np.int32)
    vals = sm[coords[:, 0], coords[:, 1], coords[:, 2]]
    return coords[np.argsort(-vals)].astype(np.int32)


def physical_nms(coords_vox: np.ndarray, scores: np.ndarray, scale: np.ndarray,
                 radius_um: float) -> tuple[np.ndarray, np.ndarray]:
    """Greedy non-max suppression in physical (micron) space."""
    if len(coords_vox) <= 1:
        return coords_vox, scores
    pts = coords_vox.astype(np.float64) * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    suppressed = np.zeros(len(coords_vox), dtype=bool)
    keep = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        for j in tree.query_ball_point(pts[i], r=radius_um):
            suppressed[j] = True
    keep = np.array(keep, dtype=np.int64)
    return coords_vox[keep], scores[keep]


def refine_centroid(vol: np.ndarray, approx_zyx: np.ndarray) -> tuple[np.ndarray, float]:
    """Intensity-weighted centroid refinement in original voxel coordinates."""
    Z, Y, X = vol.shape
    z, y, x = (int(round(v)) for v in approx_zyx)
    rz, ry, rx = 2, 5, 5
    z0, z1 = max(0, z - rz), min(Z, z + rz + 1)
    y0, y1 = max(0, y - ry), min(Y, y + ry + 1)
    x0, x1 = max(0, x - rx), min(X, x + rx + 1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)
    if crop.size == 0:
        return np.array([z, y, x], dtype=np.float64), 0.0
    bg = float(np.percentile(crop, 20.0))
    w = crop - bg
    w[w < 0] = 0
    total = float(w.sum())
    if total <= 1e-6:
        loc = np.unravel_index(int(np.argmax(crop)), crop.shape)
        return np.array([z0 + loc[0], y0 + loc[1], x0 + loc[2]], dtype=np.float64), float(crop[loc])
    zz, yy, xx = np.indices(crop.shape)
    refined = np.array([
        z0 + float((zz * w).sum() / total),
        y0 + float((yy * w).sum() / total),
        x0 + float((xx * w).sum() / total),
    ], dtype=np.float64)
    return refined, float(w.max())


def detect_cells(vol: np.ndarray, cfg: PipelineConfig,
                 prev_count: int | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Return integer centroid coords (z, y, x) and detector scores for one volume."""
    Z, Y, X = vol.shape
    scale = np.asarray(cfg.scale, dtype=np.float64)
    ds = block_mean_xy(vol, cfg.xy_ds)
    sm = gaussian_filter(ds, sigma=cfg.smooth_sigma, mode="nearest")
    threshold_abs, bg, dyn = robust_threshold(sm, cfg.thresh_rel)

    if _SKIMAGE:
        coords_ds = peak_local_max(sm, min_distance=cfg.min_peak_dist,
                                   threshold_abs=threshold_abs, exclude_border=False).astype(np.int32)
    else:
        coords_ds = _fallback_peaks(sm, threshold_abs, cfg.min_peak_dist)

    if coords_ds.size == 0:
        flat = np.argpartition(sm.ravel(), -3)[-3:]
        coords_ds = np.array(np.unravel_index(flat, sm.shape)).T.astype(np.int32)

    peak_scores = sm[coords_ds[:, 0], coords_ds[:, 1], coords_ds[:, 2]].astype(np.float32)
    rel_contrast = (peak_scores - bg) / max(dyn, 1e-6)
    keep = rel_contrast >= cfg.min_rel_contrast
    coords_ds, peak_scores = coords_ds[keep], peak_scores[keep]
    if len(coords_ds) == 0:
        return np.empty((0, 3), np.int32), np.empty((0,), np.float32)

    # XY-block grid → original coords (Z unchanged).
    approx = coords_ds.astype(np.float64)
    approx[:, 1] = approx[:, 1] * cfg.xy_ds + (cfg.xy_ds - 1) / 2.0
    approx[:, 2] = approx[:, 2] * cfg.xy_ds + (cfg.xy_ds - 1) / 2.0

    refined, refined_scores = [], []
    for a, s in zip(approx, peak_scores):
        r, rs = refine_centroid(vol, a)
        refined.append(r)
        refined_scores.append(max(float(s), rs))
    coords = np.vstack(refined).astype(np.float64)
    scores = np.array(refined_scores, dtype=np.float32)

    # Drop weak boundary peaks, keep confident border cells.
    if len(coords):
        cz, cy, cx = coords[:, 0], coords[:, 1], coords[:, 2]
        border = ((cz <= cfg.border_z) | (cz >= (Z - 1 - cfg.border_z)) |
                  (cy <= cfg.border_yx) | (cy >= (Y - 1 - cfg.border_yx)) |
                  (cx <= cfg.border_yx) | (cx >= (X - 1 - cfg.border_yx)))
        floor = float(np.quantile(scores, cfg.border_keep_quantile)) if len(scores) > 8 else -np.inf
        keep = (~border) | (scores >= floor)
        coords, scores = coords[keep], scores[keep]

    coords, scores = physical_nms(coords, scores, scale, cfg.nms_radius_um)

    # Count stabilizer: trim only implausible frame-to-frame explosions.
    if (prev_count is not None and prev_count >= 8
            and len(coords) > prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add):
        cap = int(prev_count * cfg.max_frame_count_mult + cfg.max_frame_count_add)
        order = np.argsort(-scores)[:cap]
        coords, scores = coords[order], scores[order]
    if len(coords) > cfg.max_nodes_per_frame:
        order = np.argsort(-scores)[:cfg.max_nodes_per_frame]
        coords, scores = coords[order], scores[order]

    coords = np.rint(coords).astype(np.int32)
    if len(coords):
        coords[:, 0] = np.clip(coords[:, 0], 0, Z - 1)
        coords[:, 1] = np.clip(coords[:, 1], 0, Y - 1)
        coords[:, 2] = np.clip(coords[:, 2], 0, X - 1)
    return coords, scores.astype(np.float32)

### Submission assembly & validation

In [ ]:
"""Submission assembly and validation (node/edge rows → competition CSV).

Required columns: id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
`id` is a throwaway consecutive index. node rows fill t/z/y/x (integer voxels)
with source/target = -1; edge rows fill source_id/target_id with the rest = -1.
"""

from pathlib import Path
from typing import Sequence

import pandas as pd

COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]


def node_row(dataset: str, node_id: int, t: int, zyx: Sequence[int]) -> dict:
    z, y, x = (int(v) for v in zyx)
    return {"dataset": dataset, "row_type": "node", "node_id": int(node_id),
            "t": int(t), "z": z, "y": y, "x": x, "source_id": -1, "target_id": -1}


def edge_row(dataset: str, source_id: int, target_id: int) -> dict:
    return {"dataset": dataset, "row_type": "edge", "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(source_id), "target_id": int(target_id)}


def assemble(rows: list[dict]) -> pd.DataFrame:
    """Rows → DataFrame with the exact competition columns + explicit `id` index."""
    sub = pd.DataFrame(rows)[COLUMNS]
    sub.insert(0, "id", range(len(sub)))
    return sub


def save(sub: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(path, index=False)
    return path


def validate(sub: pd.DataFrame, expected_datasets: set[str] | None = None) -> None:
    """Catch the common Kaggle submission failures before committing."""
    cols = [c for c in sub.columns if c != "id"]
    assert cols == COLUMNS, f"Wrong columns: {list(sub.columns)}"
    assert len(sub) > 0, "Empty submission"
    assert set(sub["row_type"].unique()).issubset({"node", "edge"}), "Invalid row_type"

    nodes = sub[sub.row_type == "node"]
    edges = sub[sub.row_type == "edge"]
    assert (nodes[["node_id", "t", "z", "y", "x"]] >= 0).all().all(), "Node fields must be >= 0"
    assert (nodes[["source_id", "target_id"]] == -1).all().all(), "Node source/target must be -1"
    assert (edges[["node_id", "t", "z", "y", "x"]] == -1).all().all(), "Edge node/coords must be -1"
    assert (edges[["source_id", "target_id"]] >= 0).all().all(), "Edge refs must be >= 0"

    if expected_datasets is not None:
        missing = expected_datasets - set(sub["dataset"].unique())
        assert not missing, f"Missing datasets in submission: {sorted(missing)}"

    for ds, grp in sub.groupby("dataset"):
        node_ids = set(grp.loc[grp.row_type == "node", "node_id"].astype(int))
        e = grp[grp.row_type == "edge"]
        assert node_ids, f"{ds}: no node rows"
        assert e["source_id"].astype(int).isin(node_ids).all(), f"{ds}: dangling source_id"
        assert e["target_id"].astype(int).isin(node_ids).all(), f"{ds}: dangling target_id"

### Global ILP linking (tracksdata)

In [ ]:
"""Global ILP linking (Ultrack-style) as an alternative to greedy Hungarian.

Builds a candidate-edge graph from our detections and lets `tracksdata`'s ILP
solver choose a globally consistent, flow-valid set of edges (with native ≤2-
daughter division + appear/disappear costs) — instead of locally-optimal greedy
frame-to-frame assignment. Detection is unchanged; only linking differs, so this
composes with any detector.

Offline note: the ILP uses the open-source CBC solver when no Gurobi license is
active (no internet needed). Chunking bounds solve time for large graphs.
"""

import numpy as np
import polars as pl
import tracksdata as td



def ilp_link(node_rows: list[dict], cfg: PipelineConfig,
             n_neighbors: int = 6, delta_t: int = 1,
             appearance: float = 0.1, disappearance: float = 0.1,
             division: float = 1.0, timeout: float | None = 600.0) -> list[dict]:
    """Return edge rows (source_id/target_id over node_rows' node_id) chosen by ILP.

    node_rows: dicts with node_id, t, z, y, x (voxel). Distance is computed in
    physical µm (coords scaled by cfg.scale) with an 11µm-style gate = max_link_dist_um.
    """

    if not node_rows:
        return []
    scale = np.asarray(cfg.scale, dtype=np.float64)

    g = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        g.add_node_attr_key(key, pl.Float64, -1e9)
    # store PHYSICAL coords so DistanceEdges' Euclidean distance is in µm
    gids = g.bulk_add_nodes([
        {"t": int(r["t"]),
         "z": float(r["z"]) * scale[0],
         "y": float(r["y"]) * scale[1],
         "x": float(r["x"]) * scale[2]}
        for r in node_rows
    ])
    gid_to_nodeid = {int(gid): int(r["node_id"]) for gid, r in zip(gids, node_rows)}

    # candidate edges between consecutive frames within the physical gate
    td.edges.DistanceEdges(
        distance_threshold=cfg.max_link_dist_um,
        n_neighbors=n_neighbors,
        delta_t=delta_t,   # >1 adds gap-closing candidate edges (bridge missed detections)
    ).add_edges(g)

    if g.num_edges() == 0:
        return []

    # distance -> reward: prob in [0,1], 1 at dist 0 (mirrors the reference's -1*edge_prob)
    ea = g.edge_attrs(attr_keys=[td.DEFAULT_ATTR_KEYS.EDGE_ID, "distance"])
    g.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
    prob = (1.0 - ea["distance"] / cfg.max_link_dist_um).clip(0.0, 1.0)
    g.update_edge_attrs(edge_ids=ea[td.DEFAULT_ATTR_KEYS.EDGE_ID].to_list(),
                        attrs={"edge_prob": prob.to_list()})

    solver = td.solvers.ILPSolver(
        edge_weight=-1.0 * td.EdgeAttr("edge_prob"),
        appearance_weight=appearance,
        disappearance_weight=disappearance,
        division_weight=division,
        timeout=timeout,   # bound solve time for the offline 12h budget
    )
    solved = solver.solve(g)
    if solved is None:
        # solution written in place on g under DEFAULT_ATTR_KEYS.SOLUTION
        solved = g
        sel = solved.edge_attrs(
            attr_keys=[td.DEFAULT_ATTR_KEYS.EDGE_SOURCE, td.DEFAULT_ATTR_KEYS.EDGE_TARGET,
                       td.DEFAULT_ATTR_KEYS.SOLUTION]
        ).filter(pl.col(td.DEFAULT_ATTR_KEYS.SOLUTION))
    else:
        sel = solved.edge_attrs(
            attr_keys=[td.DEFAULT_ATTR_KEYS.EDGE_SOURCE, td.DEFAULT_ATTR_KEYS.EDGE_TARGET]
        )

    dataset = node_rows[0]["dataset"]
    out = []
    for row in sel.iter_rows(named=True):
        s = gid_to_nodeid.get(int(row[td.DEFAULT_ATTR_KEYS.EDGE_SOURCE]))
        t = gid_to_nodeid.get(int(row[td.DEFAULT_ATTR_KEYS.EDGE_TARGET]))
        if s is not None and t is not None:
            out.append(edge_row(dataset, s, t))
    return out

### Stage-B post-processing (pilkwang 0.897 graph surgery, CC0)
Motion-relink + gap-close + gap2 recovery + safe divisions + short-track filter + line-fit smoothing. Adds **+0.040** on our dev subset (0.648→0.688), incl. first non-zero division score. Deps (`scipy`/`pandas`/`numpy`) are pre-installed on Kaggle — no extra wheels.

In [ ]:
"""Stage-B graph-surgery post-processing, ported from the CC0 pilkwang 0.897 pipeline.

Operates on a plain-dict graph (``nodes_by_id`` + ``edges``) decoupled from tracksdata, so it
applies to BOTH the learned geffs and our own classical submission rows. Ported near-verbatim from
``notebooks/gpu-start/pilkwang_0897_reference/cells/cell_16.py`` (CC0-1.0, pilkwang support pack);
see that folder's README for provenance and the tuned-knob table.

Adaptations for our use:
  * synthetic-midpoint image refinement is stubbed OFF (no zarr dependency); gap nodes stay geometric.
Tuned on our 13-video dev set: default stack (motion-relink + gap-close + gap2 + safe-div + short-track
+ linefit) lifts classical dev 0.6482 -> 0.6879 (+0.040). motion-relink REPLACES edges with a motion
Hungarian relink; on our (prob-less) ILP edges it still beats raw ILP. See the reference README for knobs.
Constants are module globals (pilkwang defaults); override per-run via ``apply_to_submission(df,
overrides={...})``. Not thread-safe (single-threaded pipeline use).
"""

import math
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)

# --- tuned knobs (pilkwang defaults; see reference README) ------------------
OUTPUT_ENFORCE_NEXT_FRAME = True
OUTPUT_EDGE_MAX_UM = 14.0
OUTPUT_SINGLE_PARENT_REPAIR = True
OUTPUT_SINGLE_CHILD_REPAIR = False
OUTPUT_MOTION_RELINK = True           # tuned ON: +motion Hungarian relink beats raw ILP on our dev set
MOTION_RELINK_VELOCITY_WEIGHT = 0.5
MOTION_RELINK_LEARNED_BONUS = 0.75
MOTION_RELINK_TIGHT_UM = 6.0
MOTION_RELINK_RELAXED_UM = 10.0
MOTION_RELINK_MAX_FRAME_NODES = 2600
OUTPUT_GAP_CLOSE = True
GAP_CLOSE_MAX_GAP = 1
GAP_CLOSE_UM = 6.0
GAP_CLOSE_REUSE_EXISTING = True
GAP_CLOSE_REUSE_UM = 3.2
GAP_CLOSE_MAX_ADDED_ABS = 2000
GAP_CLOSE_MAX_ADDED_FRAC = 0.05
GAP_REFINE_SYNTHETIC = False          # image refinement OFF in this port (no zarr dep)
GAP_REFINE_WIN_YX = 3
GAP_REFINE_WIN_Z = 1
GAP_REFINE_MAX_SHIFT_UM = 3.2
OUTPUT_GAP2_RECOVERY = True           # tuned ON: dt=2 gap recovery adds edge recall on our dev set
GAP2_MAX_TOTAL_UM = 10.2
GAP2_MAX_STEP_UM = 4.4
GAP2_MAX_LINKS_ABS = 180
GAP2_MAX_LINKS_FRAC = 0.0045
GAP2_FRAME_FRAC_CAP = 0.006
GAP2_REQUIRE_CONTEXT = True
OUTPUT_SAFE_DIVISIONS = True
SAFE_DIV_MAX_UM = 4.7
SAFE_DIV_SISTER_MAX_UM = 7.2
SAFE_DIV_EXISTING_CHILD_MAX_UM = 7.8
SAFE_DIV_FRAME_FRAC_CAP = 0.008
SAFE_DIV_GLOBAL_FRAC_CAP = 0.004
OUTPUT_DIVISION_GEOMETRY_FILTER = False
DIV_PARENT_MAX_UM = 10.5
DIV_SISTER_MAX_UM = 8.0
DIV_DROP_TO_SINGLE_IF_BAD = True
OUTPUT_PRUNE_ISOLATED = True
OUTPUT_FILTER_SHORT_TRACKS = True
OUTPUT_MIN_TRACK_LEN = 6
OUTPUT_KEEP_DIVISION_COMPONENTS = True
OUTPUT_LINEFIT_SMOOTH = True
OUTPUT_LINEFIT_WEIGHT = 0.8
OUTPUT_LINEFIT_WINDOW = 2


def refine_synthetic_midpoint(dataset, t, midpoint, frame_cache, stats):
    """Image refinement disabled in this port (needs zarr frames) — return geometric midpoint."""
    return midpoint


def edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def node_point(node: dict[str, object]) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    return prob_value, -float(edge["distance_um"])


def _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1

def _position_um(node: dict[str, object]) -> np.ndarray:
    return np.array(
        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],
        dtype=np.float64,
    )

def motion_relink_edges(
    nodes_by_id: dict[int, dict[str, object]],
    stats: dict[str, int],
    learned_edge_probs: dict[tuple[int, int], float] | None = None,
) -> list[dict[str, object]]:
    if not OUTPUT_MOTION_RELINK or not nodes_by_id:
        return []

    learned_edge_probs = learned_edge_probs or {}

    def learned_prob(source_id: int, target_id: int) -> float:
        value = learned_edge_probs.get((source_id, target_id), 0.0)
        try:
            value = float(value)
        except (TypeError, ValueError):
            return 0.0
        if not np.isfinite(value):
            return 0.0
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        return float(np.clip(value, 0.0, 1.0))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()

    frame_sizes = [len(ids) for ids in ids_by_t.values()]
    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}
    predecessor_position_um: dict[int, np.ndarray] = {}
    selected_edges: list[dict[str, object]] = []

    def assign_pass(
        source_ids: list[int],
        target_ids: list[int],
        gate_um: float,
    ) -> list[tuple[int, int, float, float, float]]:
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, source_id in enumerate(source_ids):
            source_pos = position_um[source_id]
            prev_pos = predecessor_position_um.get(source_id)
            if prev_pos is None:
                predicted = source_pos
            else:
                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)
            for j, target_id in enumerate(target_ids):
                target_pos = position_um[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos))
                if raw > gate_um:
                    continue
                motion = float(np.linalg.norm(target_pos - predicted))
                prob = learned_prob(source_id, target_id)
                raw_dist[i, j] = raw
                motion_dist[i, j] = motion
                prob_matrix[i, j] = prob
                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob
        row_ind, col_ind = linear_sum_assignment(cost)
        matches: list[tuple[int, int, float, float, float]] = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= big:
                continue
            matches.append((
                source_ids[int(r)],
                target_ids[int(c)],
                float(raw_dist[r, c]),
                float(motion_dist[r, c]),
                float(prob_matrix[r, c]),
            ))
        return matches

    times = sorted(ids_by_t)
    for t in times:
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue
        unmatched_sources = set(source_ids)
        unmatched_targets = set(target_ids)
        frame_matches: list[tuple[int, int, float, float, str, float]] = []
        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):
            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]
            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]
            matches = assign_pass(pass_sources, pass_targets, gate_um)
            for source_id, target_id, raw, motion, prob in matches:
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id)
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))
                if pass_name == "tight":
                    stats["motion_relink_tight_edges"] += 1
                else:
                    stats["motion_relink_relaxed_edges"] += 1
        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:
            selected_edges.append({
                "source_id": source_id,
                "target_id": target_id,
                "edge_prob": prob,
                "distance_um": raw,
                "motion_distance_um": motion,
                "motion_relinked": 1,
                "motion_pass": pass_name,
            })
            predecessor_position_um[target_id] = position_um[source_id]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected_edges)
    return selected_edges

def close_single_frame_gaps(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)

    max_synthetic = min(
        GAP_CLOSE_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,
    )
    next_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    used_starts: set[int] = set()
    used_isolated: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict[str, object]] = []

    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)
    stats["gap_close_effective_max_gap"] = effective_gap_max
    for gap in range(1, effective_gap_max + 1):
        for t, end_ids in sorted(ends_by_t.items()):
            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]
            if not end_ids or not start_ids:
                continue

            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]
            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]
            threshold_um = GAP_CLOSE_UM * (gap + 1)
            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)
            for i, ep in enumerate(end_points):
                for j, sp in enumerate(start_points):
                    d[i, j] = point_distance_um(ep, sp)
            stats["gap_candidates"] += int((d <= threshold_um).sum())
            if not np.isfinite(d).any():
                continue

            big = threshold_um * 1000.0 + 1.0
            cost = np.where(d <= threshold_um, d, big)
            row_ind, col_ind = linear_sum_assignment(cost)
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > threshold_um:
                    continue
                source_id = end_ids[int(r)]
                target_id = start_ids[int(c)]
                if source_id in outgoing or target_id in used_starts:
                    continue

                source = nodes_by_id[source_id]
                target = nodes_by_id[target_id]
                mid_t = int(source["t"]) + gap
                mid_point = (
                    (float(source["z"]) + float(target["z"])) / 2.0,
                    (float(source["y"]) + float(target["y"])) / 2.0,
                    (float(source["x"]) + float(target["x"])) / 2.0,
                )

                middle_id: int | None = None
                if GAP_CLOSE_REUSE_EXISTING:
                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]
                    if candidates:
                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]
                        best_idx = int(np.argmin(distances))
                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:
                            middle_id = candidates[best_idx]
                            used_isolated.add(middle_id)
                            stats["gap_reused_existing"] += 1

                if middle_id is None:
                    if synthetic_added >= max_synthetic:
                        stats["gap_skipped_node_cap"] += 1
                        continue
                    middle_id = next_id
                    next_id += 1
                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                    nodes_by_id[middle_id] = {
                        "node_id": middle_id,
                        "t": mid_t,
                        "z": refined_point[0],
                        "y": refined_point[1],
                        "x": refined_point[2],
                    }
                    synthetic_added += 1
                    stats["gap_inserted_synthetic"] += 1

                middle = nodes_by_id[middle_id]
                e1 = {
                    "source_id": source_id,
                    "target_id": middle_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(source, middle),
                    "gap_closed": 1,
                }
                e2 = {
                    "source_id": middle_id,
                    "target_id": target_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(middle, target),
                    "gap_closed": 1,
                }
                new_edges.extend([e1, e2])
                outgoing.add(source_id)
                incoming.add(middle_id)
                outgoing.add(middle_id)
                incoming.add(target_id)
                used_starts.add(target_id)
                stats["gap_pairs_selected"] += 1
                stats["gap_added_edges"] += 2

    if new_edges:
        edges = [*edges, *new_edges]
    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, edges

def _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_source: dict[int, list[int]] = {}
    for edge in edges:
        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))
    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}


def _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_target: dict[int, list[int]] = {}
    for edge in edges:
        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))
    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}

def recover_strict_gap2(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    predecessor = _single_predecessor_map(edges)
    successor = _single_successor_map(edges)

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)

    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))
    proposals: list[tuple[float, int, int, int, float]] = []

    def pos_um(node_id: int) -> np.ndarray:
        node = nodes_by_id[node_id]
        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)

    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = starts_by_t.get(t + 3, [])
        if not end_ids or not start_ids:
            continue
        for end_id in end_ids:
            end_pos = pos_um(end_id)
            for start_id in start_ids:
                start_pos = pos_um(start_id)
                dist = float(np.linalg.norm(start_pos - end_pos))
                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:
                    continue
                step = (start_pos - end_pos) / 3.0
                context_penalty = 0.0
                if GAP2_REQUIRE_CONTEXT:
                    ok_context = False
                    prev_id = predecessor.get(end_id)
                    if prev_id is not None:
                        prev_step = end_pos - pos_um(prev_id)
                        prev_norm = float(np.linalg.norm(prev_step))
                        step_norm = float(np.linalg.norm(step))
                        if prev_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    next_id = successor.get(start_id)
                    if next_id is not None:
                        next_step = pos_um(next_id) - start_pos
                        next_norm = float(np.linalg.norm(next_step))
                        step_norm = float(np.linalg.norm(step))
                        if next_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    if not ok_context:
                        continue
                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))

    proposals.sort(key=lambda item: item[0])
    stats["gap2_candidates"] = len(proposals)
    if not proposals:
        return nodes_by_id, edges

    selected: list[tuple[float, int, int, int, float]] = []
    used_ends: set[int] = set()
    used_starts: set[int] = set()
    per_frame_count: dict[int, int] = {}
    for proposal in proposals:
        if len(selected) >= cap:
            stats["gap2_skipped_cap"] += 1
            break
        _, end_id, start_id, t, _ = proposal
        if end_id in used_ends or start_id in used_starts:
            continue
        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))
        if per_frame_count.get(t, 0) >= frame_cap:
            continue
        selected.append(proposal)
        used_ends.add(end_id)
        used_starts.add(start_id)
        per_frame_count[t] = per_frame_count.get(t, 0) + 1

    if not selected:
        return nodes_by_id, edges

    next_node_id = _next_node_id(nodes_by_id)
    frame_cache: dict[int, np.ndarray] = {}
    new_edges: list[dict[str, object]] = []
    for _, end_id, start_id, t, _ in selected:
        source = nodes_by_id[end_id]
        target = nodes_by_id[start_id]
        previous_id = end_id
        inserted_ids: list[int] = []
        for k in (1, 2):
            frac = k / 3.0
            mid_t = int(source["t"]) + k
            midpoint = (
                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,
                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,
                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,
            )
            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)
            node_id = next_node_id
            next_node_id += 1
            nodes_by_id[node_id] = {
                "node_id": node_id,
                "t": mid_t,
                "z": refined_point[0],
                "y": refined_point[1],
                "x": refined_point[2],
            }
            inserted_ids.append(node_id)
            current = nodes_by_id[node_id]
            new_edges.append({
                "source_id": previous_id,
                "target_id": node_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),
                "gap2_recovered": 1,
            })
            previous_id = node_id
        new_edges.append({
            "source_id": previous_id,
            "target_id": start_id,
            "edge_prob": None,
            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),
            "gap2_recovered": 1,
        })
        stats["gap2_pairs_selected"] += 1
        stats["gap2_added_nodes"] += len(inserted_ids)
        stats["gap2_added_edges"] += 3

    return nodes_by_id, [*edges, *new_edges]

def add_safe_divisions_postlink(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> list[dict[str, object]]:
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges

    out_by_source: dict[int, list[dict[str, object]]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)

    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    used_targets: set[int] = set()

    for t in sorted(ids_by_t):
        child_frame_ids = ids_by_t.get(t + 1, [])
        if not child_frame_ids:
            continue
        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]
        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]
        if not source_ids or not candidate_ids:
            continue

        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float, float]] = []
        for source_id in source_ids:
            source = nodes_by_id[source_id]
            existing_child_edge = out_by_source[source_id][0]
            existing_child_id = int(existing_child_edge["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_dist = edge_distance_um(source, existing_child)
            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
            for candidate_id in candidate_ids:
                if (source_id, candidate_id) in existing_edges:
                    continue
                candidate = nodes_by_id[candidate_id]
                parent_dist = edge_distance_um(source, candidate)
                if parent_dist > SAFE_DIV_MAX_UM:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                if sister_dist > SAFE_DIV_SISTER_MAX_UM:
                    continue
                score = parent_dist + 0.15 * sister_dist
                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))

        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist, _ in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            candidate = nodes_by_id[candidate_id]
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_dist,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            added_this_frame += 1

    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges

def filter_short_track_components(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a: int, b: int) -> None:
        if a not in parent or b not in parent:
            return
        ra = find(a)
        rb = find(b)
        if ra != rb:
            parent[ra] = rb

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        union(source_id, target_id)
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    keep: set[int] = set()
    for members in components.values():
        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)
        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):
            keep.update(members)

    if not keep:
        stats["short_track_filter_skipped_all"] += 1
        return nodes_by_id, edges

    removed_nodes = len(nodes_by_id) - len(keep)
    if removed_nodes <= 0:
        return nodes_by_id, edges

    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))
    stats["short_track_nodes_removed"] = removed_nodes
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges

def linefit_smooth_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> dict[int, dict[str, object]]:
    """Smooth linear track interiors without changing graph topology."""
    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    predecessor: dict[int, list[int]] = {}
    successor: dict[int, list[int]] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        successor.setdefault(source_id, []).append(target_id)
        predecessor.setdefault(target_id, []).append(source_id)

    original_pos = {
        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)
        for node_id, node in nodes_by_id.items()
    }
    updated_pos: dict[int, np.ndarray] = {}
    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))

    for node_id in sorted(nodes_by_id):
        neighbourhood: list[tuple[int, int]] = [(0, node_id)]

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            prev_ids = predecessor.get(current, [])
            if len(prev_ids) != 1:
                break
            current = prev_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((-step, current))

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            next_ids = successor.get(current, [])
            if len(next_ids) != 1:
                break
            current = next_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((step, current))

        if len(neighbourhood) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue

        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)
        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])
        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)
        if not np.isfinite(fitted).all():
            stats["linefit_skipped_nodes"] += 1
            continue
        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted

    for node_id, pos in updated_pos.items():
        nodes_by_id[node_id]["z"] = float(pos[0])
        nodes_by_id[node_id]["y"] = float(pos[1])
        nodes_by_id[node_id]["x"] = float(pos[2])

    stats["linefit_smoothed_nodes"] = len(updated_pos)
    return nodes_by_id

def filter_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    raw_edges: list[dict[str, object]],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:
    stats = {
        "raw_edges": len(raw_edges),
        "dropped_nonconsecutive_edges": 0,
        "dropped_long_edges": 0,
        "dropped_multi_parent_edges": 0,
        "dropped_multi_child_edges": 0,
        "dropped_division_edges": 0,
        "gap_candidates": 0,
        "gap_pairs_selected": 0,
        "gap_reused_existing": 0,
        "gap_inserted_synthetic": 0,
        "gap_added_nodes": 0,
        "gap_added_edges": 0,
        "gap_skipped_node_cap": 0,
        "gap_refined_synthetic": 0,
        "gap_refine_failed": 0,
        "gap_refine_rejected_shift": 0,
        "pruned_isolated_nodes": 0,
        "motion_relink_edges": 0,
        "motion_relink_tight_edges": 0,
        "motion_relink_relaxed_edges": 0,
        "motion_relink_frames": 0,
        "motion_relink_replaced_raw_edges": 0,
        "motion_relink_fallback_raw": 0,
        "motion_relink_skipped_large_frame": 0,
        "gap2_candidates": 0,
        "gap2_pairs_selected": 0,
        "gap2_added_nodes": 0,
        "gap2_added_edges": 0,
        "gap2_skipped_cap": 0,
        "safe_division_candidates": 0,
        "safe_divisions_added": 0,
        "safe_division_skipped_cap": 0,
        "short_track_components_removed": 0,
        "short_track_nodes_removed": 0,
        "short_track_edges_removed": 0,
        "short_track_filter_skipped_all": 0,
        "linefit_smoothed_nodes": 0,
        "linefit_skipped_nodes": 0,
    }

    edges: list[dict[str, object]] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance_um = edge_distance_um(source, target)
        edge["distance_um"] = distance_um
        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge)

    if OUTPUT_MOTION_RELINK:
        learned_edge_probs: dict[tuple[int, int], float] = {}
        for edge in edges:
            prob = edge.get("edge_prob")
            if prob is None:
                continue
            try:
                prob = float(prob)
            except (TypeError, ValueError):
                continue
            if np.isfinite(prob):
                key = (int(edge["source_id"]), int(edge["target_id"]))
                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)
        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs)
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1

    if OUTPUT_SINGLE_PARENT_REPAIR and edges:
        best_by_target: dict[int, dict[str, object]] = {}
        for edge in edges:
            target_id = int(edge["target_id"])
            prev = best_by_target.get(target_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_target[target_id] = edge
        kept_ids = {id(edge) for edge in best_by_target.values()}
        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    if OUTPUT_SINGLE_CHILD_REPAIR and edges:
        best_by_source: dict[int, dict[str, object]] = {}
        for edge in edges:
            source_id = int(edge["source_id"])
            prev = best_by_source.get(source_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_source[source_id] = edge
        kept_ids = {id(edge) for edge in best_by_source.values()}
        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]

    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset)
    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)
    edges = add_safe_divisions_postlink(nodes_by_id, edges, stats)

    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:
        by_source: dict[int, list[dict[str, object]]] = {}
        for edge in edges:
            by_source.setdefault(int(edge["source_id"]), []).append(edge)

        filtered: list[dict[str, object]] = []
        for source_id, source_edges in by_source.items():
            if len(source_edges) <= 1:
                filtered.extend(source_edges)
                continue

            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)
            source = nodes_by_id[source_id]
            top1 = ranked[0]
            top2 = ranked[1]
            d1 = float(top1["distance_um"])
            d2 = float(top2["distance_um"])
            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])
            valid_division = (
                max(d1, d2) <= DIV_PARENT_MAX_UM
                and sister <= DIV_SISTER_MAX_UM
                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1
                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1
            )
            if valid_division:
                filtered.extend([top1, top2])
                stats["dropped_division_edges"] += max(0, len(ranked) - 2)
            elif DIV_DROP_TO_SINGLE_IF_BAD:
                filtered.append(top1)
                stats["dropped_division_edges"] += len(ranked) - 1
            else:
                filtered.extend(ranked)
        edges = filtered

    if OUTPUT_PRUNE_ISOLATED:
        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}
        if incident:
            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]

    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)

    return nodes_by_id, edges, stats


# --- adapters: our submission rows <-> plain-dict graph ---------------------
SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

_KNOBS = [k for k, v in list(globals().items())
          if k.isupper() and not k.startswith("_") and isinstance(v, (int, float, bool, str))]


def submission_to_graph(grp: pd.DataFrame):
    """One dataset's node/edge rows -> (nodes_by_id, raw_edges). Our edges carry no edge_prob."""
    nodes_by_id: dict[int, dict] = {}
    for r in grp[grp.row_type == "node"].itertuples():
        nid = int(r.node_id)
        nodes_by_id[nid] = {"node_id": nid, "t": int(r.t), "z": float(r.z), "y": float(r.y), "x": float(r.x)}
    raw_edges = [{"source_id": int(r.source_id), "target_id": int(r.target_id), "edge_prob": None}
                 for r in grp[grp.row_type == "edge"].itertuples()]
    return nodes_by_id, raw_edges


def graph_to_rows(nodes_by_id: dict, edges: list, dataset: str) -> list[dict]:
    rows = []
    for nid in sorted(nodes_by_id):
        n = nodes_by_id[nid]
        rows.append({"dataset": dataset, "row_type": "node", "node_id": int(n["node_id"]),
                     "t": int(n["t"]), "z": max(0, int(round(float(n["z"])))),
                     "y": max(0, int(round(float(n["y"])))), "x": max(0, int(round(float(n["x"])))),
                     "source_id": -1, "target_id": -1})
    for e in edges:
        rows.append({"dataset": dataset, "row_type": "edge", "node_id": -1, "t": -1, "z": -1, "y": -1,
                     "x": -1, "source_id": int(e["source_id"]), "target_id": int(e["target_id"])})
    return rows


def apply_to_submission(df: pd.DataFrame, overrides: dict | None = None):
    """Post-process every dataset in a submission DataFrame. Returns (new_df, stats_by_dataset).

    ``overrides`` temporarily rebinds module knobs (e.g. {"OUTPUT_MOTION_RELINK": True}); restored after.
    """
    g = globals()
    saved = {}
    if overrides:
        for k, v in overrides.items():
            if k not in _KNOBS:
                raise KeyError(f"unknown postprocess knob: {k!r}")
            saved[k] = g[k]; g[k] = v
    try:
        out, stats_by = [], {}
        for dataset, grp in df.groupby("dataset", sort=True):
            nodes_by_id, raw_edges = submission_to_graph(grp)
            nodes_by_id, edges, stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset)
            out.extend(graph_to_rows(nodes_by_id, edges, dataset))
            stats_by[dataset] = stats
    finally:
        for k, v in saved.items():
            g[k] = v
    res = pd.DataFrame(out)[SUBMISSION_COLUMNS]
    res.insert(0, "id", range(len(res)))
    return res, stats_by

### Offline image reader

In [ ]:
# --- Offline image reader (blosc2 chunk reader; verified byte-identical to zarr) ---
import json as _json

def read_array_meta(zarr_path: Path):
    candidates = [zarr_path / "0" / "zarr.json", zarr_path / "0" / ".zarray"]
    candidates += list(zarr_path.rglob("zarr.json"))[:8]
    candidates += list(zarr_path.rglob(".zarray"))[:8]
    seen = set()
    for meta_path in candidates:
        if meta_path in seen or not meta_path.exists():
            continue
        seen.add(meta_path)
        try:
            meta = _json.load(open(meta_path))
        except Exception:
            continue
        shape = tuple(meta.get("shape", ()))
        if len(shape) != 4:
            continue
        dtype = np.dtype(meta.get("data_type", meta.get("dtype")))
        if "chunk_grid" in meta:
            chunk_shape = tuple(meta["chunk_grid"]["configuration"]["chunk_shape"])
        else:
            chunk_shape = tuple(meta.get("chunks", shape))
        return meta_path.parent, shape, dtype, chunk_shape
    raise FileNotFoundError(f"Could not find 4D zarr metadata under {zarr_path}")


def _chunk_candidates(array_dir: Path, t: int):
    return [array_dir / "c" / str(t) / "0" / "0" / "0",
            array_dir / f"{t}.0.0.0", array_dir / str(t) / "0" / "0" / "0"]


def load_volume(array_dir: Path, shape, dtype, t: int) -> np.ndarray:
    import blosc2
    for cp in _chunk_candidates(array_dir, t):
        if cp.exists():
            raw = open(cp, "rb").read()
            try:
                dec = blosc2.decompress(raw)
            except Exception:
                dec = raw
            arr = np.frombuffer(dec, dtype=dtype)
            expected = int(np.prod(shape[1:]))
            if arr.size < expected:
                out = np.zeros(expected, dtype=dtype); out[:arr.size] = arr; arr = out
            return arr[:expected].reshape(shape[1:])
    raise FileNotFoundError(f"Missing chunk for t={t} in {array_dir}")

### Run → `submission.csv`

In [ ]:
# --- Run: detect every frame -> global ILP linking -> submission.csv ---------
import time, gc

# Best config (dev-subset 0.648): finer XY detection + global ILP linking.
cfg = PipelineConfig(xy_ds=2, thresh_rel=0.30, link_method="ilp",
                     ilp_appearance=0.1, ilp_disappearance=0.1, ilp_division=1.0)

def run_dataset(zarr_path: Path, dataset: str) -> list:
    array_dir, shape, dtype, _ = read_array_meta(zarr_path)
    T = shape[0]
    node_rows, node_scores = [], {}
    prev_count, next_id = None, 1
    t0 = time.time()
    for t in range(T):
        vol = load_volume(array_dir, shape, dtype, t)
        coords, scores = detect_cells(vol, cfg, prev_count=prev_count)
        del vol; gc.collect()
        if len(coords):
            o = np.lexsort((coords[:, 2], coords[:, 1], coords[:, 0])); coords, scores = coords[o], scores[o]
        ids = list(range(next_id, next_id + len(coords))); next_id += len(coords)
        for nid, zyx, sc in zip(ids, coords, scores):
            node_rows.append(node_row(dataset, nid, t, zyx)); node_scores[int(nid)] = float(sc)
        prev_count = len(coords)
    edges = ilp_link(node_rows, cfg, n_neighbors=cfg.link_n_neighbors, delta_t=cfg.link_delta_t,
                     appearance=cfg.ilp_appearance, disappearance=cfg.ilp_disappearance,
                     division=cfg.ilp_division, timeout=cfg.ilp_timeout)
    node_rows, edges, _ = prune_isolated(node_rows, edges, node_scores, cfg)
    print(f"  [{dataset}] {time.time()-t0:.0f}s | {len(node_rows)} nodes, {len(edges)} edges")
    return node_rows + edges


def prune_isolated(node_rows, edge_rows, node_scores, cfg):
    if not cfg.prune_isolated_nodes or not node_rows:
        return node_rows, edge_rows, None
    keep = set()
    for e in edge_rows:
        keep.add(int(e["source_id"])); keep.add(int(e["target_id"]))
    kn = [r for r in node_rows if int(r["node_id"]) in keep]
    kids = {int(r["node_id"]) for r in kn}
    ke = [e for e in edge_rows if int(e["source_id"]) in kids and int(e["target_id"]) in kids]
    return kn, ke, None


def resolve_test_dir() -> Path:
    for c in [Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/test"),
              Path("/kaggle/input/biohub-cell-tracking-during-development/test"),
              Path("data/test"), Path("../data/test")]:
        if c.exists() and list(c.glob("*.zarr")):
            return c
    for h in [Path(p) for p in __import__("glob").glob("/kaggle/input/**/test", recursive=True)]:
        if list(h.glob("*.zarr")):
            return h
    raise FileNotFoundError("No test/*.zarr directory found")


TEST_DIR = resolve_test_dir()
OUT = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
names = sorted(p.name[:-5] for p in TEST_DIR.iterdir() if p.name.endswith(".zarr"))
print(f"TEST_DIR={TEST_DIR}  datasets={names}")

t0 = time.time()
rows = []
for i, n in enumerate(names, 1):
    print(f"[{i}/{len(names)}] {n}")
    rows.extend(run_dataset(TEST_DIR / f"{n}.zarr", n))
sub = assemble(rows)
# Stage-B graph surgery (pilkwang 0.897, CC0): +0.040 on our dev subset (0.648 -> 0.688).
sub, _pp = apply_to_submission(sub)
validate(sub, expected_datasets=set(names))
save(sub, OUT)
print(f"\nWrote {OUT}: {len(sub):,} rows "
      f"({int((sub.row_type=='node').sum()):,} nodes, {int((sub.row_type=='edge').sum()):,} edges) "
      f"in {(time.time()-t0)/60:.1f} min")
sub.head()